In [ ]:
import os, sys
print("cwd:", os.getcwd())
print("sys.path[:5]:", sys.path[:5])

In [ ]:
%cd D:\shahnawaz\uva\main

In [ ]:
%load_ext autoreload
%autoreload 2

from src.data_loader import DataLoader
from src.data_analyzer import DataAnalyzer
from src.constant_manager import ConstantManager
from src.data_cleaner import DataCleaner
from src.feature_renamer import FeatureRenamer
from src.scaler import Scaler
from src.soed.agents.lcm_multistep_agent_gpy import LCMMultiStepAgentGpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import plotly.graph_objects as go

from src.soed.agents.multistep_mimo_agent import MultiStepMIMOAgent
from src.soed.agents.lcm_multistep_agent_gpy import LCMMultiStepAgentGpy
from src.soed.animate_plot import MultiOutputSliceAnimator

In [ ]:
# file names declared globally for easy access
RAW_DATA_FILE = 'rcci_data_v4_5.xlsx'
PD_DATA_FILE = 'rcci_cleaned_data_v4_5.parquet'

In [ ]:
pd_loader = DataLoader(file_path=PD_DATA_FILE)
pd_df = pd_loader.load_data()

pd_df.head()

In [ ]:
scaler_x = Scaler()
scaler_y = Scaler()

if_names = ConstantManager().RAW_REDUCED_INPUT_COLUMNS
of_names = ConstantManager().RAW_OUTPUT_COLUMNS

scaled_df = pd_df[if_names + of_names].copy()
scaled_df[if_names] = scaler_x.fit_transform(scaled_df, if_names)
scaled_df[of_names] = scaler_y.fit_transform(scaled_df, of_names)


In [ ]:
scaled_df.head()

In [ ]:
case1 = scaled_df.iloc[0]
print("Case 1 (scaled):")
print(case1)

In [ ]:
case1_inputs_original = scaler_x.inverse_transform(case1[if_names], dim_idx=range(len(if_names)))
print("\nCase 1 Inputs (original scale): {}".format(case1_inputs_original))
    
case1_outputs_original = scaler_y.inverse_transform(case1[of_names], dim_idx=range(len(of_names)))
print("\nCase 1 Outputs (original scale): {}".format(case1_outputs_original))

In [ ]:
data_analyzer = DataAnalyzer(scaled_df)
estimated_noise_levels_scaled, estimated_residuals_scaled = data_analyzer.estimate_noise_levels(if_names, of_names)
print("Estimated noise levels in output features after scaling:", estimated_noise_levels_scaled)
print("Estimated residuals in output features after scaling:", estimated_residuals_scaled)

In [ ]:
import torch
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from torch.quasirandom import SobolEngine

from src.soed.dynamic_gp import DynamicGP

from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.exceptions import ModelFittingError
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll
import gpytorch

warnings.filterwarnings("ignore")
plt.style.use("bmh")
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)

# ==========================================
# 2. Multi-Step Path Planner (Unchanged)
# ==========================================
class MultiStep_sOED_Agent:
    def __init__(self, bounds):
        self.bounds = bounds
        self.d = bounds.shape[1] 
        self.model, self.likelihood = None, None
        self.X, self.Y = None, None

    def fit_data(self, X, Y):
        if isinstance(X, (pd.DataFrame, pd.Series)): X = X.values
        if isinstance(Y, (pd.DataFrame, pd.Series)): Y = Y.values
        
        self.X = torch.as_tensor(X, dtype=torch.float64)
        self.Y = torch.as_tensor(Y, dtype=torch.float64).view(-1, 1)

        # self.likelihood = gpytorch.likelihoods.GaussianLikelihood(
        #     noise_constraint=gpytorch.constraints.Interval(1e-4, 1)
        # )

        self.likelihood = gpytorch.likelihoods.GaussianLikelihood(
            noise_constraint=gpytorch.constraints.Interval(1e-3, 1.0)
        )
        
        # self.likelihood.noise = torch.tensor(0.00154353)
        # self.likelihood.raw_noise.requires_grad = False  

        self.model = DynamicGP(self.X, self.Y, self.likelihood, self.bounds)
        
        self.model.train()
        self.likelihood.train()
        mll = ExactMarginalLogLikelihood(self.likelihood, self.model)
        
        with gpytorch.settings.cholesky_jitter(1e-4):
            try:
                fit_gpytorch_mll(mll)
            except ModelFittingError:
                print("Warning: L-BFGS-B optimizer failed. Falling back to Adam...")
                optimizer = torch.optim.Adam(self.model.parameters(), lr=0.05)
                for _ in range(150):
                    optimizer.zero_grad()
                    output = self.model(*self.model.train_inputs)
                    loss = -mll(output, self.model.train_targets)
                    loss.backward()
                    optimizer.step()
        
        self.model.eval()
        self.likelihood.eval()

        # --- ADD THESE LINES TO PRINT THE LEARNED PARAMETERS ---
        with torch.no_grad():
            # Get the output scale (variance of the data overall)
            output_scale = self.model.covar_module.outputscale.item()
            
            # Get the lengthscales (base_kernel is the RBFKernel inside the ScaleKernel)
            lengthscales = self.model.covar_module.base_kernel.lengthscale.squeeze().tolist()
            
            # Get the learned noise
            noise = self.likelihood.noise.item()

            print("\n--- Model Fitting Complete ---")
            print(f"Learned Output Scale: {output_scale:.4f}")
            print(f"Learned Noise: {noise:.6f}")
            if isinstance(lengthscales, list):
                for i, ls in enumerate(lengthscales):
                    print(f"Learned Lengthscale for Feature {i} ({self.bounds.shape[1]} total): {ls:.4f}")
            else:
                print(f"Learned Lengthscale: {lengthscales:.4f}")
            print("------------------------------\n")

    def plan_multistep_batch(self, current_location, q_steps=3, num_scenarios=200, w_dist=1.0):
        sobol = SobolEngine(dimension=self.d * q_steps, scramble=True)
        raw_samples = sobol.draw(num_scenarios)
        
        paths_01 = raw_samples.view(num_scenarios, q_steps, self.d)
        range_x = self.bounds[1] - self.bounds[0]
        paths = self.bounds[0] + (range_x * paths_01)

        noise_var = self.likelihood.noise.item()

        with torch.no_grad():
            post = self.model.posterior(paths)
            covars = post.distribution.covariance_matrix
            
            I = torch.eye(q_steps, dtype=covars.dtype, device=covars.device)
            matrix_to_det = I + (covars / noise_var)
            
            try:
                Ls = torch.linalg.cholesky(matrix_to_det)
                igs = Ls.diagonal(dim1=-2, dim2=-1).log().sum(dim=-1)
            except RuntimeError:
                igs = 0.5 * torch.linalg.slogdet(matrix_to_det)[1]

            curr_loc = current_location.squeeze()
            d_start = torch.norm(paths[:, 0, :] - curr_loc, dim=-1)
            
            if q_steps > 1:
                d_steps = torch.norm(paths[:, 1:, :] - paths[:, :-1, :], dim=-1).sum(dim=-1)
            else:
                d_steps = torch.zeros_like(d_start)
                
            total_dist = d_start + d_steps
            scores = igs - (w_dist * total_dist)

            max_idx = torch.argmax(scores)
            best_path = paths[max_idx]

        return best_path

# ==========================================
# 3. Static Recommendation Dashboard
# ==========================================
def plot_recommendations(agent, optimal_path, bounds, feature_names):
    """Generates a static 1x2 plot of the GP Mean, Variance, and suggested next steps."""
    res = 50 
    d = bounds.shape[1]
    
    # Grid for visualization (Assumes first 2 dimensions for plotting)
    X1, X2 = torch.meshgrid(
        torch.linspace(bounds[0, 0].item(), bounds[1, 0].item(), res),
        torch.linspace(bounds[0, 1].item(), bounds[1, 1].item(), res), indexing="xy"
    )

    x_grid = torch.zeros(res * res, d)
    x_grid[:, 0] = X1.flatten()
    x_grid[:, 1] = X2.flatten()
    # If d > 2, keep the other dimensions at their baseline (0 or mean) for the surface plot
    
    with torch.no_grad():
        post = agent.model.posterior(x_grid)
        mean = post.mean.squeeze(-1).numpy().reshape(res, res)
        var = post.variance.squeeze(-1).numpy().reshape(res, res)

    fig = plt.figure(figsize=(24, 12))
    gs = gridspec.GridSpec(1, 2, wspace=0.1) 
    ax_mean = fig.add_subplot(gs[0, 0], projection='3d')
    ax_var = fig.add_subplot(gs[0, 1], projection='3d')

    f0, f1 = feature_names[0], feature_names[1]
    curr_loc = agent.X[-1].numpy()
    path = optimal_path.numpy()
    
    # Connect current location to the planned path
    full_path_X = np.vstack([curr_loc, path])
    path_Z_var = [var.max()] * len(full_path_X)

    # --- LEFT (Mean) ---
    ax_mean.plot_surface(X1.numpy(), X2.numpy(), mean, cmap='viridis', alpha=0.4, edgecolor='none')
    ax_mean.scatter(agent.X[:, 0].numpy(), agent.X[:, 1].numpy(), agent.Y.flatten().numpy(), 
                    c='k', s=40, label="Historical Data")
    ax_mean.set_title(f"Predictive Mean Surface\n{f0} vs {f1}")

    # --- RIGHT (Variance) ---
    ax_var.plot_surface(X1.numpy(), X2.numpy(), var, cmap='plasma', alpha=0.6, edgecolor='none')
    ax_var.plot(full_path_X[:, 0], full_path_X[:, 1], path_Z_var, c='darkgreen', linestyle='--', linewidth=2, label="Suggested Path")
    ax_var.scatter(path[:, 0], path[:, 1], [var.max()]*len(path), c='orange', s=100, edgecolors='k', label="Suggested Experiments")
    ax_var.scatter(path[0, 0], path[0, 1], var.max(), c='r', marker='*', s=300, edgecolors='none', label="Do This Next")
    ax_var.set_title("Predictive Variance & Suggested Next Steps")

    for ax in [ax_mean, ax_var]:
        # ax.view_init(elev=55, azim=-70)
        ax.view_init(elev=35, azim=-45)
        # ax.legend(loc="upper left")
        ax.set_xlabel(f0); ax.set_ylabel(f1)
        ax.set_facecolor('white')
        ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
        # ax.grid(False)

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. Main Execution
# ==========================================
if __name__ == "__main__":
    # Ensure your bounds match your exact input feature dimensions
    # E.g., if you have 2 features, bounds should be shape [2, 2]
    my_bounds = torch.tensor([[-2.0, -2.0], [3.0, 3.0]]) 
    input_features = ['Boost pressure', 'Mass1']
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df[input_features]
    output = scaled_df['IEMP']
    # -----------------------------------------------------

    print("1. Initializing Agent and fitting historical data...")
    agent = MultiStep_sOED_Agent(my_bounds)
    agent.fit_data(inputs, output)
    
    print("2. Calculating the most informative next experiments...")
    # Assume the last row of your inputs is the "current state" of the system
    current_loc = agent.X[-1:] 
    
    # Get the next 3 recommended steps
    q_horizon = 3
    optimal_path = agent.plan_multistep_batch(current_location=current_loc, q_steps=q_horizon, w_dist=1.5)
    
    print("\n--- RECOMMENDED NEXT EXPERIMENTS ---")
    for step_idx, setting in enumerate(optimal_path):
        print(f"Step {step_idx + 1}: {setting.numpy()}")
        
    print("\n3. Rendering visualization...")
    plot_recommendations(agent, optimal_path, my_bounds, input_features)

In [ ]:
import torch
import pandas as pd
import plotly.graph_objects as go

from src.soed.agents.multistep_mimo_agent import MultiStepMIMOAgent
from src.soed.animate_plot import MultiOutputSliceAnimator

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
%matplotlib widget

# =========================================================
# Example Usage
# =========================================================
if __name__ == "__main__":

    my_bounds = torch.tensor([[-2.0, -2.0, -2.0, -2.0], [3.0, 3.0, 3.0, 3.0]]) 
    input_features = ['Boost pressure', 'Mass1', 'Mass2', 'IVO']
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df[input_features]
    output = scaled_df[['IEMP', 'Nox']]

    agent = MultiStepMIMOAgent(my_bounds)
    agent.fit_data(inputs.to_numpy(), output.to_numpy())

    current_loc = agent.X[-1:]
    path = agent.plan_multistep_batch(current_loc, q_steps=3, w_dist=1.5)
    
    # inverse transform the path to original feature space for better interpretability
    inverse_transformed_path = []
    for p in path:
        original_values = []
        for dim_idx, dim_name in enumerate(input_features):
            original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
            original_values.append(original_value)
            print(f"{dim_name}: {original_value:.4f}", end=" | ")
        inverse_transformed_path.append(original_values)
        print()

    # for p in path:
    #     for dim_idx, dim_name in enumerate(input_features):
    #         original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
    #         print(f"{dim_name}: {original_value:.4f}", end=" | ")
    #     print()

    # animated_plot(agent, path, output_idx=0)

    
    
    anim = MultiOutputSliceAnimator(
        agent,
        path,
        dim_x=1,
        dim_y=2,
        fixed_at="current"
    )
    plt.ion()
    anim.animate(save_path="soed_path.gif", fps=2)



    
    # animated_plot_slice(
    #     agent,
    #     path,
    #     output_idx=0,
    #     dim_x=2,  # Mass2
    #     dim_y=3   # IVO
    # )
    
    # for out_idx, name in enumerate(['IEMP', 'NOx']):
    #     animated_plot_slice(agent, path, output_idx=out_idx)


In [ ]:
import plotly.express as px

df_path = pd.DataFrame(
    inverse_transformed_path,
    columns=input_features
)

fig = px.parallel_coordinates(
    df_path,
    dimensions=input_features,
    color=df_path.index
)

fig.show()


In [ ]:
import torch
import pandas as pd
import plotly.graph_objects as go

from src.soed.agents.multistep_mimo_agent import MultiStepMIMOAgent
from src.soed.animate_plot import MultiOutputSliceAnimator

torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
%matplotlib widget

# =========================================================
# Example Usage
# =========================================================
if __name__ == "__main__":

    my_bounds = torch.tensor([[-2.0, -2.0, -2.0, -2.0, -2.0, -2.0, -2.0, -2.0, -2.0], [3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]]) 
    input_features = ConstantManager().RAW_REDUCED_INPUT_COLUMNS
    output_features = ConstantManager().RAW_OUTPUT_COLUMNS
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df[input_features]
    output = scaled_df[output_features]

    agent = MultiStepMIMOAgent(my_bounds)
    agent.fit_data(inputs.to_numpy(), output.to_numpy())

    current_loc = agent.X[-1:]
    path = agent.plan_multistep_batch(current_loc, q_steps=3, w_dist=1.5)
    
    # inverse transform the path to original feature space for better interpretability
    inverse_transformed_path = []
    for p in path:
        original_values = []
        for dim_idx, dim_name in enumerate(input_features):
            original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
            original_values.append(original_value)
            print(f"{dim_name}: {original_value:.4f}", end=" | ")
        inverse_transformed_path.append(original_values)
        print()

    # for p in path:
    #     for dim_idx, dim_name in enumerate(input_features):
    #         original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
    #         print(f"{dim_name}: {original_value:.4f}", end=" | ")
    #     print()

    # animated_plot(agent, path, output_idx=0)



    
    # animated_plot_slice(
    #     agent,
    #     path,
    #     output_idx=0,
    #     dim_x=2,  # Mass2
    #     dim_y=3   # IVO
    # )
    
    # for out_idx, name in enumerate(['IEMP', 'NOx']):
    #     animated_plot_slice(agent, path, output_idx=out_idx)


In [ ]:
scaler_x_4 = Scaler()
scaler_y_4 = Scaler()

if_names_4 = ['Boost pressure', 'Mass1', 'Mass2', 'IVO']

scaled_df_4 = pd_df[if_names_4].copy()
scaled_df_4[if_names_4] = scaler_x_4.fit_transform(scaled_df_4, if_names_4)

In [ ]:
unscaled_bounds = ConstantManager().UNSCALED_BOUNDS

unscaled_bounds_4 = {feature: unscaled_bounds[feature] for feature in if_names_4}
unscaled_bounds_4

In [ ]:
scaled_bounds_transformed = scaler_x_4.transform(torch.tensor(list(unscaled_bounds_4.values())).T)
scaled_bounds = scaled_bounds_transformed.T
scaled_bounds

In [ ]:


torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
%matplotlib widget

# =========================================================
# Example Usage
# =========================================================
if __name__ == "__main__":


    found_bounds = torch.tensor(scaled_bounds) #find_bounds(scaled_df_4, if_names_4)
    # my_bounds = torch.stack([found_bounds[:, 0], found_bounds[:, 1]], dim=0)
    # my_bounds = torch.tensor([[-2.0, -2.0, -2.0, -2.0], [3.0, 3.0, 3.0, 3.0]]) 
    my_bounds = found_bounds.T

    input_features = if_names_4
    output_features = ConstantManager().RAW_OUTPUT_COLUMNS


    mass_idx_1 = input_features.index("Mass1")
    mass_idx_2 = input_features.index("Mass2")
    
    # for idx in [mass_idx_1, mass_idx_2]:
    #     # Calculate what 0.0 kg maps to in the scaled space
    #     # Formula: Z = (X - mean) / scale
    #     mean_val = scaler_x_4.scaler.mean_[idx]
    #     scale_val = scaler_x_4.scaler.scale_[idx]
    #     scaled_zero = (0.0 - mean_val) / scale_val
        
    #     # Update the lower bound (my_bounds[0]) to be AT LEAST scaled_zero
    #     # This prevents SobolEngine from ever proposing a point below 0 kg
    #     my_bounds[0, idx] = torch.max(my_bounds[0, idx], torch.tensor(scaled_zero, dtype=torch.float64))

    print(f"Bounds after ensuring non-negativity for Mass1 and Mass2: {my_bounds[0]} to {my_bounds[1]}")

    # print the bounds in original scale for better interpretability
    print("\nBounds in original scale:")
    for idx, feature in enumerate(input_features):
        original_lower = scaler_x_4.inverse_transform(my_bounds[0, idx].item(), idx)
        original_upper = scaler_x_4.inverse_transform(my_bounds[1, idx].item(), idx)
        print(f"{feature}: [{original_lower:.4f}, {original_upper:.4f}]")
    
    # 1. Initialize with 5 starting points
    print("Gathering initial historical points...")
    inputs = scaled_df_4[input_features]
    output = scaled_df[output_features]

    agent = MultiStepMIMOAgent(my_bounds)
    agent.fit_data(inputs.to_numpy(), output.to_numpy())

    # print details about the fitted model
    print("\n--- Model Fitting Complete ---")
    for idx, model in enumerate(agent.models):
        print("\n --------------------------------------------------")
        print(f"OutputModel {idx}: {output_features[idx]}")
        print(f"Learned Output Scale: {model.covar_module.outputscale.item():.4f}")
        print(f"Learned Noise: {model.likelihood.noise.item():.6f}")
        lengthscales = model.covar_module.base_kernel.lengthscale.squeeze().tolist()
        print(f"Learned Lengthscales: {lengthscales}")
        print(" --------------------------------------------------")

    current_loc = agent.X[-1:]
    path = agent.plan_multistep_batch(current_loc, q_steps=3, w_dist=1.5)
    
    # inverse transform the path to original feature space for better interpretability
    inverse_transformed_path = []
    for p in path:
        original_values = []
        for dim_idx, dim_name in enumerate(input_features):
            original_value = scaler_x_4.inverse_transform(p[dim_idx].item(), dim_idx)
            original_values.append(original_value)
            print(f"{dim_name}: {original_value:.4f}", end=" | ")
        inverse_transformed_path.append(original_values)
        print()

    # for p in path:
    #     for dim_idx, dim_name in enumerate(input_features):
    #         original_value = scaler_x_4.inverse_transform(p[dim_idx].item(), dim_idx)
    #         print(f"{dim_name}: {original_value:.4f}", end=" | ")
    #     print()

    # animated_plot(agent, path, output_idx=0)



    
    # animated_plot_slice(
    #     agent,
    #     path,
    #     output_idx=0,
    #     dim_x=2,  # Mass2
    #     dim_y=3   # IVO
    # )
    
    # for out_idx, name in enumerate(['IEMP', 'NOx']):
    #     animated_plot_slice(agent, path, output_idx=out_idx)


In [ ]:
inverse_transformed_path = []
for p in path:
    original_values = []
    for dim_idx, dim_name in enumerate(input_features):
        original_value = scaler_x_4.inverse_transform(p[dim_idx].item(), dim_idx)
        original_values.append(original_value)
        print(f"{dim_name}: {original_value:.10f}", end=" | ")
    inverse_transformed_path.append(original_values)
    print()

#### Planning on trained model

In [ ]:
import torch
import gpytorch

# Load the dictionary
saved_data = torch.load('surrogate_model_likelihood_and_data.pth', weights_only=False)

# Extract your training data first
Xtr = torch.tensor(saved_data['Xtr'])
Ytr = torch.tensor(saved_data['Ytr'])

In [ ]:
class LCM_MultitaskExactGP(gpytorch.models.ExactGP):
    """
    Multi‑output Exact GP using LCMKernel (Linear Model of Coregionalization).
    - input kernel = Matern(ν=1.5, ARD) + RationalQuadratic(ARD)
    - task coupling via LMC with NUM_LATENTS latent processes
    Expects:
       train_x: (N, D)
       train_y: (N, T)
    """
    def __init__(self, train_x, train_y, likelihood, num_tasks, input_dim, num_latents=2):
        super().__init__(train_x, train_y, likelihood)
        self.num_tasks = num_tasks

        # Mean: one const mean per task
        # self.mean_module = gpytorch.means.MultitaskMean(
        #     gpytorch.means.ConstantMean(), num_tasks=num_tasks
        # )

        self.mean_module = gpytorch.means.MultitaskMean(
            gpytorch.means.LinearMean(input_size=input_dim), num_tasks=num_tasks
        )

        # Base kernels over inputs
        rbf = gpytorch.kernels.RBFKernel(ard_num_dims=input_dim, lengthscale_prior=gpytorch.priors.SmoothedBoxPrior(0.1, 10.0))
        matern = gpytorch.kernels.MaternKernel(nu=0.5, ard_num_dims=input_dim)
        matern15 = gpytorch.kernels.MaternKernel(nu=1.5, ard_num_dims=input_dim)
        rq     = gpytorch.kernels.RQKernel(ard_num_dims=input_dim)

        # LCM kernel: combine base kernels into latent processes
        # LCMKernel handles the task coregionalization internally
        self.covar_module = gpytorch.kernels.LCMKernel(
            base_kernels=[matern, rq],  # can add more kernels here
            num_tasks=num_tasks,
            rank=num_latents    # latent rank (num_latents)
        )

    def forward(self, x):
        mean_x = self.mean_module(x)      # (N, T)
        covar_x = self.covar_module(x)    # multitask covariance
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)
    
    def get_covar_module(self):
        return self.covar_module

In [ ]:
device = "cpu"
T = Ytr.shape[1]


likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(
    num_tasks=T, 
    has_task_noise=True,
    rank=0,
    noise_constraint=gpytorch.constraints.GreaterThan(1e-4)
).to(device)

model = LCM_MultitaskExactGP(Xtr, Ytr, likelihood, num_tasks=T, input_dim=9, num_latents=11).to(device)


model.load_state_dict(saved_data['model_state'])
likelihood.load_state_dict(saved_data['likelihood_state'])

# Put them in eval mode if you are doing inference
# model.eval()
# likelihood.eval()

In [ ]:
torch.tensor(list(unscaled_bounds.values()))

In [ ]:
unscaled_bounds_9 = {feature: unscaled_bounds[feature] for feature in if_names}

scaled_bounds_transformed = scaler_x.transform(torch.tensor(list(unscaled_bounds_9.values())).T)
scaled_bounds = scaled_bounds_transformed.T
scaled_bounds

In [ ]:
Xtr.shape

In [ ]:
Xtr[-1]

In [ ]:
planner_A = LCMMultiStepAgentGpy(bounds=scaled_bounds, model=model, likelihood=likelihood)

# Assume your current engine state is represented by this vector
current_state = Xtr[-1]

# Plan the next 3 steps
next_experiments = planner_A.plan_multistep_batch(
    current_location=current_state,
    q_steps=3,
    num_scenarios=800, # You can bump this up; vectorization handles it easily
    w_dist=1.0
)

print(next_experiments)

In [ ]:
inverse_transformed_path = []
for p in next_experiments:
    original_values = []
    for dim_idx, dim_name in enumerate(if_names):
        original_value = scaler_x.inverse_transform(p[dim_idx].item(), dim_idx)
        original_values.append(original_value)
        # print(f"{dim_name}: {original_value:.10f}", end=" | ")
        print(f"{original_value:.10f}", end="\t")

    inverse_transformed_path.append(original_values)
    print()

In [ ]:
import pandas as pd

df = pd.DataFrame(inverse_transformed_path, columns=if_names)

df.to_clipboard(index=False, sep="\t")

In [ ]:
import matplotlib.pyplot as plt
import torch

%matplotlib inline

# --- CONFIGURATION ---
# Select which of the 0-8 inputs represent your physical 2D map
X_IDX = 0 
Y_IDX = 1

# Extract historical path
history_x = Xtr[:, X_IDX].cpu().numpy()
history_y = Xtr[:, Y_IDX].cpu().numpy()

# Extract planned 3-step path
planned_x = next_experiments[:, X_IDX].cpu().numpy()
planned_y = next_experiments[:, Y_IDX].cpu().numpy()

plt.figure(figsize=(8, 6))
plt.scatter(history_x, history_y, c='lightgray', label='Past Data (198 points)', alpha=0.5)

# Plot current location (the very last point in Xtr)
plt.plot(history_x[-1], history_y[-1], 'bo', markersize=8, label='Current Location') 

# Plot the planner's proposed future steps
plt.plot(planned_x, planned_y, 'r-x', linewidth=2, markersize=8, label='Planned 3 Steps')

plt.title(f"Future Experiment Projected Path (Input {X_IDX} vs Input {Y_IDX})")
plt.xlabel(f"Feature {X_IDX}")
plt.ylabel(f"Feature {Y_IDX}")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# --- CONFIGURATION ---
HISTORY_LOOKBACK = 5  # How many past steps to show for context
FUTURE_STEPS = 3

# 1. Get the real historical outputs for the last few steps
# Assuming Ytr shape is [198, 11]
history_steps = min(HISTORY_LOOKBACK, Ytr.shape[0])
past_means = Ytr[-history_steps:].cpu().numpy()

# 2. Evaluate the 3 planned steps through your surrogate model
model.eval()
with torch.no_grad():
    predictions = model(next_experiments)
    pred_means = predictions.mean.cpu().numpy()  # Shape: [3, 11]
    pred_vars = predictions.variance.cpu().numpy()

# 3. Set up the X-axis timelines
# History will be negative steps (-5, -4, -3, -2, -1, 0), Future will be (1, 2, 3)
# Step 0 represents the "Current Location" absolute transition point
history_timeline = np.arange(-history_steps + 1, 1) 
future_timeline = np.arange(1, FUTURE_STEPS + 1)

# Plotting
plt.figure(figsize=(14, 7))
colors = plt.cm.tab20(np.linspace(0, 1, 11))

for task_idx in range(11):
    # A. Plot Historical Ground Truth (Dashed Line)
    plt.plot(history_timeline, past_means[:, task_idx], 
             linestyle='--', 
             marker='x', 
             color=colors[task_idx], 
             alpha=0.7, 
             linewidth=1.5)
    
    # B. Connect history to future by stitching the last historical point to the first prediction
    bridge_x = [0, 1]
    bridge_y = [past_means[-1, task_idx], pred_means[0, task_idx]]
    plt.plot(bridge_x, bridge_y, 
             linestyle=':', 
             color=colors[task_idx], 
             alpha=0.5)

    # C. Plot Planned Future Predictions (Solid Line)
    plt.plot(future_timeline, pred_means[:, task_idx], 
             linestyle='-', 
             marker='o', 
             color=colors[task_idx], 
             linewidth=2.5,
             label=f'Response {task_idx}')

# Add a vertical divider at step 0 to visually split Past vs Future
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3, linewidth=1.5)
plt.text(-1.5, plt.ylim()[0] + (plt.ylim()[1]-plt.ylim()[0])*0.9, 
         'PAST (Observed)', fontsize=10, color='gray', weight='bold')
plt.text(0.01, plt.ylim()[0] + (plt.ylim()[1]-plt.ylim()[0])*0.9, 
         'FUTURE (Planned)', fontsize=10, color='crimson', weight='bold')

plt.title("Response Trajectories: Comparison of Observed History vs. Planned Future in Scaled Space")
plt.xlabel("Experiments Timeline (Relative to Current State)")
plt.ylabel("Response Value")

# Combine timelines for clean X-ticks labeling
all_ticks = list(history_timeline) + list(future_timeline)
plt.xticks(all_ticks, [f"ex{i}" if i <= 0 else f"ex+{i}" for i in all_ticks])

plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Extract lengthscales
lss = planner_A.effective_lengthscales()
if isinstance(lss, list):
    lss = torch.stack(lss).mean(dim=0).to(current_state.device)

# Combine current state with the 3 planned steps to get a 4-step sequence
full_sequence = torch.cat([current_state.unsqueeze(0), next_experiments], dim=0)

print("--- Distance Verification ---")
for i in range(len(full_sequence) - 1):
    point_a = full_sequence[i]
    point_b = full_sequence[i+1]
    
    # Calculate the Mahalanobis distance between consecutive steps
    dist = torch.sqrt(((point_a - point_b) / lss).pow(2).sum())
    print(f"Step {i} -> {i+1} Distance: {dist.item():.4f}")
    
    if dist > 1.5: # Or whatever your max step threshold is
        print(f"  [WARNING] Distance exceeds threshold of {1.5}!")

In [ ]:
def get_path_variance(path_tensor):
    with torch.no_grad():
        preds = model(path_tensor)
        # Sum the variance across all 3 steps and all 11 tasks
        return preds.variance.sum().item()

# 1. Variance of the CHOSEN path
chosen_var = get_path_variance(next_experiments)

# 2. Variance of a RANDOM path (just add some noise to current state)
random_path = current_state.unsqueeze(0).repeat(3, 1) + torch.randn(3, 9).to(current_state.device) * 0.1
random_var = get_path_variance(random_path)

print("--- Optimality Verification ---")
print(f"Chosen Path Total Variance: {chosen_var:.4f}")
print(f"Random Path Total Variance: {random_var:.4f}")

if chosen_var > random_var:
    print("Success: The planner successfully found a path with higher uncertainty/reward than random guessing.")